# E-commerce Customer Analytics — Analysis Walkthrough
**Business question:** Who are our most valuable customers, how well do we retain them, and where should marketing spend go next?

**Data:** UCI Online Retail — 541,909 invoice lines from a UK online gift retailer (Dec 2010 – Dec 2011).
All transformations are written in **SQL** (`sql/` folder, run on DuckDB); Python is used for orchestration and visualisation.

In [1]:
import duckdb, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.max_columns', 20); pd.set_option('display.width', 160)
con = duckdb.connect('../data/retail.duckdb', read_only=True)
q = lambda s: con.sql(s).df()

## 1. Data quality audit
Every cleaning rule is logged so the impact is transparent. Rows without a CustomerID can't be used for customer-level analysis; cancelled orders, postage/fee lines and purchases later reversed by a cancellation are removed.

In [2]:
q('SELECT * FROM data_quality_log')

,check_name,rows
0,Raw rows,541909
1,Missing CustomerID,135080
2,Cancellation lines (C-invoices),9288
3,Non-product codes (postage/fees),2950
4,Zero / negative price,2517
5,"Negative qty, not cancellation",1336
6,Exact duplicate lines,5270
7,Purchases reversed by a cancellation,4032
8,Clean rows kept,387230


## 2. Headline KPIs and monthly trend
Month-over-month growth and cumulative revenue are computed with SQL window functions (`LAG`, running `SUM`).

In [3]:
display(q('SELECT * FROM headline_kpis'))
q('SELECT * FROM monthly_kpis')

,total_revenue,total_orders,total_customers,total_products,countries,avg_order_value,orders_per_customer
0,8261197.0,18247,4324,3645,37,452.74,4.22


,month,revenue,orders,active_customers,units,avg_order_value,mom_growth_pct,cumulative_revenue,is_partial_month
0,2010-12-01,553412.96,1385,883,305550.0,399.58,NaN,553412.96,False
1,2011-01-01,453929.65,972,736,260791.0,467.01,-18.0,1007342.61,False
2,2011-02-01,436239.31,986,755,262239.0,442.43,-3.9,1443581.92,False
3,2011-03-01,569355.83,1299,968,341566.0,438.30,30.5,2012937.75,False
4,2011-04-01,447197.16,1127,848,288332.0,396.80,-21.5,2460134.91,False
5,2011-05-01,649583.35,1526,1050,367643.0,425.68,45.3,3109718.26,False
6,2011-06-01,602524.41,1374,986,354994.0,438.52,-7.2,3712242.67,False
7,2011-07-01,580010.81,1313,945,360824.0,441.74,-3.7,4292253.48,False
8,2011-08-01,621376.29,1262,933,389763.0,492.37,7.1,4913629.77,False
9,2011-09-01,923760.68,1724,1252,536970.0,535.82,48.7,5837390.45,False


![monthly](../outputs/charts/01_monthly_revenue.png)

**Insight:** revenue more than doubles from the Feb-2011 low to the Nov-2011 peak — Sep–Nov is the critical trading window (gift season). December 2011 only has 9 days of data, so it is excluded from trend conclusions.

## 3. RFM segmentation
Customers are scored 1–5 on Recency, Frequency and Monetary value using `NTILE(5)`, then mapped to business segments.

In [4]:
q('SELECT * FROM rfm_segment_summary')

,segment,customers,pct_customers,revenue,pct_revenue,avg_recency_days,avg_orders,avg_spend
0,Champions,991,22.9,5623566.0,68.1,13.0,10.6,5675.0
1,Loyal Customers,999,23.1,1223383.0,14.8,33.0,3.7,1225.0
2,Hibernating / Lost,1125,26.0,469074.0,5.7,220.0,1.1,417.0
3,At Risk (High Value),206,4.8,371608.0,4.5,131.0,5.5,1804.0
4,About to Sleep,398,9.2,333131.0,4.0,150.0,2.4,837.0
5,Need Attention,371,8.6,166068.0,2.0,54.0,1.2,448.0
6,New / Promising,234,5.4,74366.0,0.9,19.0,1.0,318.0


In [5]:
q("SELECT customer_id, country, recency_days, frequency, monetary, rfm_code FROM rfm WHERE segment='At Risk (High Value)' ORDER BY monetary DESC LIMIT 10")

,customer_id,country,recency_days,frequency,monetary,rfm_code
0,16180,United Kingdom,101,8,10254.18,255
1,13093,United Kingdom,276,8,7832.47,155
2,12980,United Kingdom,158,9,7374.90,255
3,16745,United Kingdom,87,17,7180.70,255
4,13027,United Kingdom,114,6,6912.00,255
5,16182,United Kingdom,73,4,6617.65,245
6,15939,United Kingdom,90,14,6100.01,255
7,14101,United Kingdom,74,6,5976.79,255
8,16553,United Kingdom,164,12,5668.82,255
9,17061,United Kingdom,74,11,5116.13,255


![rfm](../outputs/charts/02_rfm_segments.png)

**Insight:** ~22% of customers (Champions) generate ~67% of revenue. The *At Risk (High Value)* list above is a ready-made target list for a win-back campaign.

## 4. Cohort retention

In [6]:
pv = q('SELECT * FROM cohort_retention WHERE cohort_month < DATE \'2011-12-01\'').pivot(index='cohort_month', columns='month_number', values='retention_pct')
pv.iloc[:, :8]

month_number,0,1,2,3,4,5,6,7
cohort_month,,,,,,,,
2010-12-01,100.0,36.5,32.3,38.1,35.9,39.8,36.0,34.8
2011-01-01,100.0,21.5,26.8,22.9,31.9,28.7,24.9,24.2
2011-02-01,100.0,18.6,18.6,28.3,27.0,24.4,25.5,27.0
2011-03-01,100.0,14.4,24.9,19.6,22.4,16.9,26.4,23.1
2011-04-01,100.0,21.3,20.3,21.0,19.7,23.0,21.3,26.0
2011-05-01,100.0,19.4,17.3,17.3,20.8,23.0,26.5,9.2
2011-06-01,100.0,17.4,15.7,26.4,23.1,33.1,9.5,NaN
2011-07-01,100.0,17.6,20.3,22.5,27.3,11.2,NaN,NaN
2011-08-01,100.0,20.1,23.7,24.3,12.4,NaN,NaN,NaN


In [7]:
display(q('SELECT * FROM repeat_rate'))
q('SELECT month_number, ROUND(AVG(retention_pct),1) AS avg_retention_pct FROM cohort_retention WHERE month_number BETWEEN 1 AND 6 GROUP BY 1 ORDER BY 1')

,repeat_customer_pct,repeat_revenue_pct
0,65.2,93.4


,month_number,avg_retention_pct
0,1,20.4
1,2,22.0
2,3,23.2
3,4,24.5
4,5,25.0
5,6,24.3


![cohort](../outputs/charts/03_cohort_retention.png)

**Insight:** only ~20% of new customers come back the next month, but repeat customers produce >90% of revenue — improving early retention (welcome series, second-order discount) has the largest upside.

## 5. Products, countries and cancellations

In [8]:
display(q('SELECT description, ROUND(revenue) revenue, units, buyers, cum_revenue_pct FROM product_pareto LIMIT 10'))
display(q('SELECT * FROM country_summary LIMIT 10'))
q('SELECT * FROM cancellations_summary')

,description,revenue,units,buyers,cum_revenue_pct
0,REGENCY CAKESTAND 3 TIER,135975.0,11818.0,871,1.65
1,WHITE HANGING HEART T-LIGHT HOLDER,93882.0,34174.0,853,2.78
2,JUMBO BAG RED RETROSPOT,80334.0,43377.0,633,3.75
3,PARTY BUNTING,67392.0,14957.0,707,4.57
4,ASSORTED COLOUR BIRD ORNAMENT,56310.0,35202.0,676,5.25
5,RABBIT NIGHT LIGHT,51114.0,27087.0,447,5.87
6,CHILLI LIGHTS,45861.0,9557.0,204,6.43
7,PAPER CHAIN KIT 50'S CHRISTMAS,41031.0,14983.0,610,6.92
8,BLACK RECORD COVER FRAME,38956.0,11377.0,137,7.39
9,SMALL POPCORN HOLDER,37044.0,48938.0,406,7.84


,country,revenue,customers,orders,avg_order_value,revenue_per_customer,pct_revenue
0,United Kingdom,6796137.0,3909,16445,413.26,1739.0,82.27
1,Netherlands,282162.0,9,92,3066.98,31351.0,3.42
2,EIRE,242438.0,3,252,962.06,80813.0,2.93
3,Germany,203637.0,94,443,459.68,2166.0,2.46
4,France,182005.0,87,377,482.77,2092.0,2.20
5,Australia,137049.0,9,52,2635.56,15228.0,1.66
6,Spain,51976.0,29,87,597.42,1792.0,0.63
7,Switzerland,51892.0,21,44,1179.37,2471.0,0.63
8,Japan,37314.0,8,19,1963.91,4664.0,0.45
9,Belgium,36814.0,25,98,375.65,1473.0,0.45


,cancelled_value,gross_value,cancel_rate_pct
0,474403.0,8761067.0,5.41


![pareto](../outputs/charts/04_product_pareto.png)
![heatmap](../outputs/charts/05_weekday_hour_heatmap.png)

## 6. Market basket analysis (pure SQL)
Self-join on invoice to count product pairs; support, confidence and lift computed in SQL.

In [9]:
q('SELECT * FROM basket_rules WHERE product_a < product_b ORDER BY baskets_together DESC LIMIT 15')

,product_a,product_b,baskets_together,support_pct,confidence_pct,lift
0,JUMBO BAG PINK POLKADOT,JUMBO BAG RED RETROSPOT,529,2.90,62.0,7.20
1,ALARM CLOCK BAKELIKE GREEN,ALARM CLOCK BAKELIKE RED,524,2.87,67.6,14.23
2,GREEN REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER,519,2.84,76.3,18.35
3,LUNCH BAG PINK POLKADOT,LUNCH BAG RED RETROSPOT,515,2.82,55.9,8.02
4,LUNCH BAG RED RETROSPOT,LUNCH BAG SUKI DESIGN,515,2.82,40.5,7.12
5,LUNCH BAG BLACK SKULL.,LUNCH BAG RED RETROSPOT,508,2.78,49.0,7.04
6,LUNCH BAG BLACK SKULL.,LUNCH BAG SUKI DESIGN,461,2.53,44.5,7.81
7,LUNCH BAG RED RETROSPOT,LUNCH BAG SPACEBOY DESIGN,461,2.53,36.3,6.73
8,GARDENERS KNEELING PAD CUP OF TEA,GARDENERS KNEELING PAD KEEP CALM,460,2.52,72.9,17.71
9,LUNCH BAG BLACK SKULL.,LUNCH BAG PINK POLKADOT,460,2.52,44.4,8.79


## 7. Recommendations
1. **Protect Champions** — loyalty tier / early access to Q4 ranges; they are ~67% of revenue.
2. **Win back At-Risk high-value customers** — targeted reactivation offer to the exported list.
3. **Fix first-90-day retention** — welcome journey + second-purchase incentive; month-1 retention is only ~20%.
4. **Plan for Q4** — Sep–Nov ≈ 38% of annual revenue; inventory and staffing should ramp from August.
5. **Bundle high-lift pairs** — 'frequently bought together' widgets for teacups, lunch bags, alarm clocks.
6. **Diversify beyond the UK** — 82% revenue concentration; Netherlands, EIRE, Germany and France already show high revenue per customer.